# `Pipeline` - Orchestrateur du flux de données kidas

**Module :** `kadi.kidas.pipeline`  
**Classe :** `Pipeline`

---

## Introduction

`Pipeline` est le **point d'entrée unique** du module kidas. Il orchestre l'ensemble du flux :

```
Source (CSV/Excel/JSON/NetCDF/API)
         ↓
    DataCleaner
         ↓
   DataValidator
         ↓
  DataNormalizer
         ↓
     Cache
         ↓
   DataFrame final + rapport
```

Son **API fluide chainable** permet de définir un pipeline complexe en quelques lignes.  
L'**auto-détection de source** reconnaît l'extension du fichier ou le préfixe `https://`.

## 1. Préparation des données de test

In [ ]:
import os
import pandas as pd

import os
import pandas as pd

import os
import tempfile
import pandas as pd
import numpy as np

from kadi.kidas.pipeline import Pipeline
from kadi.kidas.cache import Cache

# Répertoire de cache temporaire pour les exemples
cache_dir = os.path.join(tempfile.gettempdir(), 'kidas_pipeline_demo')

# --- Création d'un fichier CSV de démonstration avec anomalies ---
df_brut = pd.DataFrame({
    'culture':      ['Maïs', 'Niébé', 'igname', 'MAÏS', None, 'sorgho'],
    'marche':       ['Dantokpa', 'Parakou', 'Bohicon', 'Dantokpa', 'Kandi', 'Natitingou'],
    'prix_xof':     [350, 500, 250, 350, 300, 280],
    'rendement_kg': [1500.0, 800.0, 2000.0, 1500.0, np.nan, 1200.0],
    'lat':          [6.366, 9.337, 7.181, 6.366, 11.133, 10.296],
    'lon':          [2.437, 2.629, 2.067, 2.437, 2.940, 1.387],
})

fichier_csv = os.path.join(tempfile.gettempdir(), 'recoltes_pipeline.csv')
df_brut.to_csv(fichier_csv, index=False)

fichier_excel = os.path.join(tempfile.gettempdir(), 'recoltes_pipeline.xlsx')
df_brut.to_excel(fichier_excel, index=False)

print(f"Fichier CSV   : {fichier_csv}")
print(f"Fichier Excel : {fichier_excel}")

## 2. Utilisation rapide - `load_clean()`

Pour un usage simple, la fonction `load_clean()` du module kidas crée automatiquement un pipeline standard.

In [ ]:
import kadi.kidas as kidas

# Une seule ligne pour charger et nettoyer
df_rapide, rapport_rapide = kidas.load_clean(fichier_csv, cache=False)

print(f"Résultat : {len(df_rapide)} lignes × {len(df_rapide.columns)} colonnes")
df_rapide

## 3. `add_source()` - Configuration de la source

**Auto-détection des types :**

| Extension | Type détecté |
|---|---|
| `.csv`, `.tsv`, `.txt` | `csv` |
| `.xls`, `.xlsx`, `.xlsm` | `excel` |
| `.json` | `json` |
| `.nc`, `.nc4`, `.netcdf` | `netcdf` |
| `http://`, `https://` | `api` |

In [ ]:
# Chargement depuis un fichier CSV (auto-détection)
pipeline = Pipeline()
pipeline._cache = Cache(cache_dir=cache_dir)
pipeline.add_source(fichier_csv)

print(f"Configuration du pipeline :")
config = pipeline.config()
print(f"  Source : {config['source']['path']} (type={config['source']['type']})")
print(f"  Étapes configurées : {config['nb_steps']}")

In [ ]:
# Chargement depuis un fichier Excel (auto-détection)
pipeline_excel = Pipeline()
pipeline_excel._cache = Cache(cache_dir=cache_dir)
pipeline_excel.add_source(fichier_excel)
config_excel = pipeline_excel.config()
print(f"Type détecté pour .xlsx : {config_excel['source']['type']}")

## 4. `add_step()` - Ajout d'étapes de nettoyage

In [ ]:
pipeline = Pipeline()
pipeline._cache = Cache(cache_dir=cache_dir)

# Ajout de plusieurs étapes de nettoyage (chainable)
pipeline.add_source(fichier_csv)

# Étape 1 : suppression des doublons
pipeline.add_step('drop_dupes')

# Étape 2 : imputation des NaN par la médiane
pipeline.add_step('fill_missing', strategy='median', cols=['rendement_kg'])

# Étape 3 : standardisation du texte
pipeline.add_step('norm_text', cols=['culture'], case='lower')

config = pipeline.config()
print(f"Étapes de nettoyage configurées : {config['nb_steps']}")
for etape in config['steps']:
    print(f"  - {etape['nom']:30s} | params={etape['params']}")

## 5. `validate()` - Ajout d'une étape de validation

In [ ]:
# Ajout d'une validation de schéma au pipeline
schema = {
    'culture': 'str',
    'marche':  'str',
    'prix_xof': 'float',
    'rendement_kg': 'float',
}

pipeline.validate(schema)

config = pipeline.config()
print(f"Total d'étapes (avec validation) : {config['nb_steps']}")

## 6. `add_normalization_step()` - Ajout d'une étape de normalisation

In [ ]:
# Normalisation des noms de cultures et des marchés
mappings = {
    'crops':   'culture',  # Normalise la colonne 'culture' vers FAO
    'markets': 'marche',   # Normalise la colonne 'marche' vers marchés Bénin
}

pipeline.add_step("std_crops", col=mappings["crops"]).add_step("std_markets", col=mappings["markets"])

config = pipeline.config()
print(f"Total d'étapes (avec normalisation) : {config['nb_steps']}")

## 7. `execute()` - Exécution complète du pipeline

In [ ]:
# Exécution sans cache (premier run)
df_final, rapport = pipeline.run(cache=False)

print(f"=== Résultats ===")
print(f"DataFrame final : {len(df_final)} lignes × {len(df_final.columns)} colonnes")
print(f"Lignes initiales → finales : {rapport['nb_rows_out']}")
print(f"Étapes appliquées : {rapport.get('steps_summary', [])}")
print(f"Cache utilisé     : {rapport.get('cache_utilise', False)}")
df_final

In [ ]:
# Exécution avec cache : 2ème run → lecture directe du cache
pipeline2 = Pipeline()
pipeline2._cache = Cache(cache_dir=cache_dir)

df_cached, rapport_v2 = (
    pipeline2
    .add_source(fichier_csv)
    .add_step('drop_dupes')
    .add_step('fill_missing', strategy='mean')
    .run(cache=True)
)

print(f"Cache utilisé : {rapport_v2.get('cache_utilise', False)}")
print(f"Lignes : {len(df_cached)}")

## 8. Rapport complet du pipeline

In [ ]:
print("=== Rapport Complet ===")

print("\nSource :")
if rapport['details'].get('source'):
    for k, v in rapport['details']['source'].items():
        print(f"  {k} : {v}")

print("\nNettoyage :")
if rapport['details'].get('nettoyage'):
    nett = rapport['details']['nettoyage']
    print(f"  Doublons supprimés : {nett.get('doublons_supprimes', 0)}")
    print(f"  NaN traités        : {nett.get('nan_traites', 0)}")

print("\nValidation :")
if rapport['details'].get('validation'):
    val = rapport['details']['validation']
    print(f"  Validations : {len(val.get('validations', []))}")

if rapport.get('quality_score'):
    score = rapport['quality_score']
    print(f"\nScore qualité global : {score['overall']:.2%}")

## 9. `config()` - Configuration du pipeline

In [ ]:
config = pipeline.config()
print("Configuration du pipeline :")
print(f"  Source       : {config['source']}")
print(f"  Nb étapes    : {config['nb_steps']}")
for i, etape in enumerate(config['steps'], 1):
    print(f"  Étape {i} : [{etape['type']}] {etape['nom']} → {etape['params']}")

## 10. `export()` - Export du rapport en JSON ou HTML

In [ ]:
import json

# Export au format JSON
chemin_json = os.path.join(tempfile.gettempdir(), 'rapport_kidas.json')
succes_json = pipeline.export(chemin_json)
print(f"Export JSON réussi : {succes_json}")

# Affichage d'un extrait du rapport JSON
with open(chemin_json, encoding='utf-8') as f:
    rapport_json = json.load(f)

print("\nClés du rapport JSON :")
for cle in rapport_json.keys():
    print(f"  - {cle}")

In [ ]:
# Export au format HTML
chemin_html = os.path.join(tempfile.gettempdir(), 'rapport_kidas.html')
succes_html = pipeline.export(chemin_html)
print(f"Export HTML réussi  : {succes_html}")
print(f"Rapport HTML disponible : {chemin_html}")

## 11. Pipelines avancés - Exemples complets

In [ ]:
# === Pipeline complet pour les prix de marchés ===
pipeline_marches = Pipeline()
pipeline_marches._cache = Cache(cache_dir=cache_dir)

df_marches, rapport_marches = (
    pipeline_marches
    .add_source(fichier_csv)
    .add_step('drop_dupes')
    .add_step('fill_missing', strategy='median')
    .add_step('norm_text', cols=['culture'], case='lower')
    .validate({'culture': 'str', 'prix_xof': 'float'})
    .add_step("std_crops", col="culture")
    .run(cache=False)
)

print(f"Pipeline marchés : {len(df_marches)} lignes")
df_marches[['culture', 'marche', 'prix_xof', 'rendement_kg']].head()

In [ ]:
# === Utilisation avec DataSource directe ===
from kadi.kidas.sources.csv_source import CSVSource

source_directe = CSVSource(fichier_csv, encoding='utf-8')

pipeline_direct = Pipeline()
pipeline_direct._cache = Cache(cache_dir=cache_dir)

df_direct, _ = (
    pipeline_direct
    .add_source(source_directe)  # DataSource directe (pas de chemin)
    .add_step('drop_dupes')
    .run(cache=False)
)

print(f"Via DataSource directe : {len(df_direct)} lignes")

## 12. `_detecter_type_source()` - Détection du type de source

In [ ]:
# Méthode statique : peut être utilisée indépendamment
exemples = [
    'recoltes.csv',
    'prix_marches.xlsx',
    'donnees_fao.json',
    'chirps_benin.nc',
    'https://api.open-meteo.com/forecast',
]

print("Détection automatique du type de source :")
for chemin in exemples:
    try:
        type_detecte = Pipeline._detect_kind(chemin)
        print(f"  {chemin:40s} → {type_detecte}")
    except Exception as e:
        print(f"  {chemin:40s} → ERREUR : {e}")

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `add_source(source, **kwargs)` | Configure la source (auto-détection du type) |
| `add_step(name, **params)` | Ajoute une étape DataCleaner |
| `validate(schema)` | Ajoute une validation DataValidator |
| `add_step("std_crops", col=mappings["crops"]).add_step("std_markets", col=mappings["markets"])` | Ajoute une normalisation DataNormalizer |
| `execute(cache)` | Lance le pipeline et retourne (df, rapport) |
| `config()` | Retourne la configuration complète |
| `export(filepath)` | Exporte le rapport en JSON ou HTML |
| `_detecter_type_source(path)` | Détecte le type depuis l'extension ou URL |

---

> **Conseil :** Pour les scripts de production, utilisez `cache=True` et définissez le répertoire cache dans `~/.kadi/kidas_cache/`. Cela évite de recharger des données déjà traitées lors des relances.